# Jupyter Executor

本文档演示如何在 Runtime 容器中管理和使用 Jupyter/IPython 会话。 Nexus 提供了完整的 IPython Kernel 管理能力：
1. **创建会话**：使用 `create_ipython_session` 创建独立的 Python 执行环境
2. **执行代码**：使用 `execute_ipython` 在会话中执行 Python 代码
3. **销毁会话**：使用 `destroy_ipython_session` 清理会话资源

## 初始化环境

首先导入必要的模块并配置日志。

In [1]:
from nexus.utils.log import setup_logging
from nexus.runtime import DockerRuntimeProvider

setup_logging()

## 创建并获取 Runtime 实例

创建 Docker Runtime Provider 并获取一个 Ubuntu 24.04 的运行时实例。

In [2]:
rp = DockerRuntimeProvider()
runtime = await rp.acquire(name='main', image='ubuntu:24.04')

2026-01-13 02:53:00 | INFO | nexus.runtime.docker | Image nexus-runtime:0.8.6 already exists, skipping build
2026-01-13 02:53:01 | INFO | nexus.runtime.docker | Runtime container nexus-runtime-0-8-6 already exists
2026-01-13 02:53:01 | INFO | nexus.runtime.docker | Starting container nexus-user-main...
2026-01-13 02:53:04 | INFO | nexus.runtime.docker | Runtime main ready at http://172.17.0.1:33159


## 创建 IPython 会话（create_ipython_session）

**方法签名**

```python
async def create_ipython_session(
    *,
    python_executable: str | None = None,
) -> CreateIPythonSessionResponse
```

**参数说明**

- `python_executable` (`str | None`)：要使用的 Python 可执行文件路径，必须已安装 `ipykernel`。 如果不指定，使用 Runtime 服务内置的 Python

**返回值**

`CreateIPythonSessionResponse` 包含：
- `session_id` (`str`)：创建的会话 ID（UUID 格式）

**实现位置**

- Executor 实现：`nexus/extensions/executors/jupyter.py:65-71`
- Mixin 方法：`nexus/runtime/executor_mixin.py:154-171`

In [3]:
# 创建 IPython 会话（使用默认 Python）
session = await runtime.create_ipython_session()
print(f"Created IPython session: {session.session_id}")

2026-01-13 02:53:04 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.demo.tools' with 2 registrations:
2026-01-13 02:53:04 | INFO | nexus.core.registry |   - [tool] demo/think, demo/bash
2026-01-13 02:53:04 | INFO | nexus.core.registry | Imported 'nexus.extensions.agents.swe_agent.tools' with 32 registrations:
2026-01-13 02:53:04 | INFO | nexus.core.registry |   - [executor] create_browser_session, destroy_browser_session, browser_goto, browser_go_back, browser_go_forward, browser_click, browser_fill, browser_scroll, browser_press_key, browser_hover, browse [... (truncated log)] l_session, run_command_in_terminal_session, start_command_in_terminal_session, query_terminal_command_status, send_keys_to_terminal_session, capture_terminal_pane, terminate_terminal_session_processes
2026-01-13 02:53:04 | INFO | nexus.core.registry |   - [tool] swe_agent/bash, swe_agent/str_replace_editor, swe_agent/review_submit
2026-01-13 02:53:05 | INFO | nexus.core.registry | Imported 'nexu

## 执行 Python 代码（execute_ipython）

**方法签名**

```python
async def execute_ipython(
    session_id: str,
    code: str,
    *,
    timeout: float = 180,
) -> ExecuteIPythonResponse
```

**参数说明**

- `session_id` (`str`)：会话 ID
- `code` (`str`)：要执行的 Python 代码
- `timeout` (`float`)：执行超时时间（秒），默认 180 秒

**返回值**

`ExecuteIPythonResponse` 包含：
- `status` (`str`)：执行状态（`ok`、`error`、`TimeLimitExceeded`）
- `stdout` (`str`)：标准输出内容
- `stderr` (`str`)：标准错误内容
- `display` (`list[dict]`)：显示数据（如图表、HTML 等）
- `error` (`list[dict]`)：错误信息（包含 traceback）
- `execution_time` (`float | None`)：代码执行时长（秒）

**实现位置**

- Executor 实现：`nexus/extensions/executors/jupyter.py:83-89`
- Mixin 方法：`nexus/runtime/executor_mixin.py:191-212`

### 基本代码执行

In [4]:
# 执行简单的 print 语句
result = await runtime.execute_ipython(
    session_id=session.session_id,
    code='print("Hello from IPython!")'
)

print(f"Status: {result.status}")
print(f"Output: {result.stdout}")
print(f"Execution time: {result.execution_time:.3f}s")

Status: ok
Output: Hello from IPython!

Execution time: 0.005s


### 状态保持：变量在会话中持久化

In [5]:
# 第一次执行：定义变量
result1 = await runtime.execute_ipython(
    session_id=session.session_id,
    code='x = 42\nprint(f"x = {x}")'
)
print("First execution:")
print(result1.stdout)

# 第二次执行：使用之前定义的变量
result2 = await runtime.execute_ipython(
    session_id=session.session_id,
    code='y = x * 2\nprint(f"y = {y}")'
)
print("\nSecond execution (using previous variable):")
print(result2.stdout)

First execution:
x = 42


Second execution (using previous variable):
y = 84



### 执行多行代码

In [6]:
# 执行多行代码（定义函数并调用）
code = '''
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

result = [fibonacci(i) for i in range(10)]
print(f"Fibonacci sequence: {result}")
'''

result = await runtime.execute_ipython(
    session_id=session.session_id,
    code=code
)
print(result.stdout)

Fibonacci sequence: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]



### 处理执行错误

In [7]:
# 执行会产生错误的代码
result = await runtime.execute_ipython(
    session_id=session.session_id,
    code='1 / 0  # Division by zero'
)

print(f"Status: {result.status}")
if result.error:
    print(f"Error occurred:")
    for err in result.error:
        print(f"  Type: {err.get('ename')}")
        print(f"  Message: {err.get('evalue')}")
        if 'traceback' in err:
            print(f"  Traceback:")
            for line in err['traceback'][:3]:  # 只显示前3行
                print(f"    {line}")

Status: error
Error occurred:
  Type: ZeroDivisionError
  Message: division by zero
  Traceback:
    ---------------------------------------------------------------------------
    ZeroDivisionError                         Traceback (most recent call last)
    Cell In[6], line 1
----> 1 1 / 0  # Division by zero



### 使用第三方库

In [8]:
# 使用 NumPy（假设已安装）
code = '''
try:
    import numpy as np
    arr = np.array([1, 2, 3, 4, 5])
    print(f"Array: {arr}")
    print(f"Mean: {arr.mean()}")
    print(f"Sum: {arr.sum()}")
except ImportError:
    print("NumPy not installed in this environment")
'''

result = await runtime.execute_ipython(
    session_id=session.session_id,
    code=code
)
print(result.stdout)

NumPy not installed in this environment



### 处理标准错误输出

In [9]:
# 输出到 stderr
code = '''
import sys
print("This goes to stdout")
print("This goes to stderr", file=sys.stderr)
'''

result = await runtime.execute_ipython(
    session_id=session.session_id,
    code=code
)
print(f"STDOUT: {result.stdout}")
print(f"STDERR: {result.stderr}")

STDOUT: This goes to stdout

STDERR: This goes to stderr



### 超时处理

In [10]:
# 执行一个会超时的长时间运行任务
result = await runtime.execute_ipython(
    session_id=session.session_id,
    code='import time\ntime.sleep(10)\nprint("Done")',
    timeout=2.0  # 2秒超时
)

print(f"Status: {result.status}")
print(f"Execution time: {result.execution_time:.3f}s")
if result.status == 'TimeLimitExceeded':
    print("Task was interrupted due to timeout")

Status: TimeLimitExceeded
Execution time: 2.001s
Task was interrupted due to timeout


## 销毁 IPython 会话（destroy_ipython_session）

**方法签名**

```python
async def destroy_ipython_session(
    session_id: str,
) -> DestroyIPythonSessionResponse
```

**参数说明**

- `session_id` (`str`)：要销毁的会话 ID

**返回值**

`DestroyIPythonSessionResponse` 包含：
- `success` (`bool`)：是否成功销毁

**实现位置**

- Executor 实现：`nexus/extensions/executors/jupyter.py:74-80`
- Mixin 方法：`nexus/runtime/executor_mixin.py:173-189`

In [11]:
# 销毁会话
result = await runtime.destroy_ipython_session(session_id=session.session_id)
print(f"Session destroyed: {result.success}")

Session destroyed: True


## 管理多个会话

In [12]:
# 创建两个独立的会话
session1 = await runtime.create_ipython_session()
session2 = await runtime.create_ipython_session()

# 在第一个会话中定义变量
await runtime.execute_ipython(
    session_id=session1.session_id,
    code='data = "Session 1 data"'
)

# 在第二个会话中定义同名变量
await runtime.execute_ipython(
    session_id=session2.session_id,
    code='data = "Session 2 data"'
)

# 验证两个会话的数据是独立的
result1 = await runtime.execute_ipython(
    session_id=session1.session_id,
    code='print(data)'
)
result2 = await runtime.execute_ipython(
    session_id=session2.session_id,
    code='print(data)'
)

print(f"Session 1: {result1.stdout.strip()}")
print(f"Session 2: {result2.stdout.strip()}")

# 清理会话
await runtime.destroy_ipython_session(session_id=session1.session_id)
await runtime.destroy_ipython_session(session_id=session2.session_id)

Session 1: Session 1 data
Session 2: Session 2 data


DestroyIPythonSessionResponse(success=True)

## 完整工作流示例：数据分析任务

In [13]:
# 1. 创建会话
analysis_session = await runtime.create_ipython_session()

# 2. 准备数据
await runtime.execute_ipython(
    session_id=analysis_session.session_id,
    code='''
data = [
    {'name': 'Alice', 'score': 85},
    {'name': 'Bob', 'score': 92},
    {'name': 'Charlie', 'score': 78},
    {'name': 'David', 'score': 95},
]
print("Data loaded")
'''
)

# 3. 数据分析
result = await runtime.execute_ipython(
    session_id=analysis_session.session_id,
    code='''
scores = [d['score'] for d in data]
avg_score = sum(scores) / len(scores)
max_score = max(scores)
min_score = min(scores)

print(f"Average Score: {avg_score:.2f}")
print(f"Max Score: {max_score}")
print(f"Min Score: {min_score}")
'''
)
print("Analysis Results:")
print(result.stdout)

# 4. 清理会话
await runtime.destroy_ipython_session(session_id=analysis_session.session_id)

Analysis Results:
Average Score: 87.50
Max Score: 95
Min Score: 78



DestroyIPythonSessionResponse(success=True)

## 实现位置

- Jupyter Executor 实现：`nexus/extensions/executors/jupyter.py`
- Mixin 方法：`nexus/runtime/executor_mixin.py:152-212`
- Kernel 实现：基于 `jupyter_client.AsyncKernelManager`

## 清理资源

使用完毕后释放所有 Runtime 实例。

In [14]:
await rp.release_all()

2026-01-13 02:53:13 | INFO | nexus.runtime.docker | Stopping container nexus-user-main...
2026-01-13 02:53:13 | INFO | nexus.runtime.docker | Runtime main released
